# Markdown RAG with LlamaIndex
Retrieve relevant sections of an employee handbook and generate answers.

In [1]:
import os
from pathlib import Path

from get_api_key import get_api_key
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.readers.file import FlatReader

os.environ["OPENAI_API_KEY"] = get_api_key()

In [2]:
# Configure LlamaIndex settings
Settings.llm = OpenAI(model="gpt-3.5-turbo", max_tokens=100)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

In [3]:
in_doc = 'employee_handbook.md'
parser = MarkdownNodeParser()


def parse_nodes(text_doc=in_doc):
    if not text_doc:
        return None
    documents = FlatReader().load_data(Path(text_doc))
    return parser.get_nodes_from_documents(documents)

In [4]:
# Vector index, retriever and query engine
nodes = parse_nodes()
index = VectorStoreIndex(nodes)
retriever = index.as_retriever(similarity_top_k=3)
query_engine = RetrieverQueryEngine(retriever=retriever)

2026-10-11 13:29:57,735 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [5]:
def retrive_parts(user_query):
    if not user_query or not user_query.strip():
        return None
    return retriever.retrieve(user_query)


def query_doc(user_query):
    if not user_query or not user_query.strip():
        return None
    return query_engine.query(user_query)

In [6]:
queries = [
    "What are the core work hours?",
    "What benefits do full-time employees get?",
    "What happens after a written warning?",
    "",
]

for q in queries:
    print("query:", repr(q))
    parts = retrive_parts(q)
    response = query_doc(q)
    if parts is None or response is None:
        print("retrieved:", parts, "| response:", response, "\n")
        continue
    print("retrieved:", [p.node.node_id for p in parts])
    print("response:", response)
    print("src-nodes:", [(round(s.score, 2), s.node.node_id, s.node.get_content()[:60]) for s in response.source_nodes])
    print()

query: 'What are the core work hours?'


2026-10-11 13:30:17,553 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-11 13:30:18,686 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-11 13:30:20,050 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


retrieved: ['51d02e6d-f5fe-416c-b72c-bf293eab6b8c', '841a02e3-6c63-4eef-971b-22d41e702c92', 'c6bdc0a2-43d1-4f5b-ab17-0a4aa9ae1600']
response: The core work hours are from 10:00 AM to 4:00 PM.
src-nodes: [(0.85, '51d02e6d-f5fe-416c-b72c-bf293eab6b8c', '## Work Hours\n\nThe standard work week is Monday to Friday, 4'), (0.76, '841a02e3-6c63-4eef-971b-22d41e702c92', '## Employee Benefits\n\nFull-time employees receive health, de'), (0.74, 'c6bdc0a2-43d1-4f5b-ab17-0a4aa9ae1600', '## General Queries\n\nQuestions about this handbook can be sen')]

query: 'What benefits do full-time employees get?'


2026-10-11 13:30:20,509 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-11 13:30:20,962 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-11 13:30:22,786 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


retrieved: ['841a02e3-6c63-4eef-971b-22d41e702c92', '51d02e6d-f5fe-416c-b72c-bf293eab6b8c', 'c6bdc0a2-43d1-4f5b-ab17-0a4aa9ae1600']
response: Full-time employees receive health, dental, and vision insurance starting on their first day. They are entitled to 20 days of paid vacation per year, 10 paid sick days, and a 401(k) plan with a 4% employer match. Additionally, they have access to an annual learning budget of $1,000 for courses and conferences.
src-nodes: [(0.85, '841a02e3-6c63-4eef-971b-22d41e702c92', '## Employee Benefits\n\nFull-time employees receive health, de'), (0.8, '51d02e6d-f5fe-416c-b72c-bf293eab6b8c', '## Work Hours\n\nThe standard work week is Monday to Friday, 4'), (0.75, 'c6bdc0a2-43d1-4f5b-ab17-0a4aa9ae1600', '## General Queries\n\nQuestions about this handbook can be sen')]

query: 'What happens after a written warning?'


2026-10-11 13:30:23,145 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-11 13:30:23,596 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-11 13:30:25,070 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


retrieved: ['a2564033-0272-4d4b-831a-05eecce8050f', '841a02e3-6c63-4eef-971b-22d41e702c92', 'c6bdc0a2-43d1-4f5b-ab17-0a4aa9ae1600']
response: After a written warning, the next step in handling policy violations is a final warning.
src-nodes: [(0.8, 'a2564033-0272-4d4b-831a-05eecce8050f', '## Disciplinary Measures\n\nPolicy violations are handled in s'), (0.72, '841a02e3-6c63-4eef-971b-22d41e702c92', '## Employee Benefits\n\nFull-time employees receive health, de'), (0.72, 'c6bdc0a2-43d1-4f5b-ab17-0a4aa9ae1600', '## General Queries\n\nQuestions about this handbook can be sen')]

query: ''
retrieved: None | response: None 

